# Time-varying rates

`Time()` is a first-class rate node. Structural interpolators in
`summer4.timevarying` turn knots into calibratable rate expressions —
both **x** (breakpoints) and **y** (values) may be `FieldRef`s.


## `Time()` ramp

Exit rate `Time() * 0.1` integrates $\dot y = -0.1\,t\,y$ to
$y(t) = \exp(-0.05 t^2)$.


In [ ]:
import numpy as np

from summer4 import (
    Compartments,
    ExitFlow,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Time,
    TransitionFlow,
)
from summer4.timevarying import linear, sigmoidal, step

state = Property("state", ("Y",))
pmap = PropertyMap.from_property(state)
model = FlowModel(pmap)
model.add_flow(ExitFlow("out", state["Y"], Time() * 0.1))
cm = model.compile()
y0 = PropertyData.wrap(pmap, np.array([1.0]))
t1 = 5.0
plan = SavePlan(requests={"y": SaveRequest(Compartments())}, ts=np.linspace(0.0, t1, 51))
res = cm.run({}, y0, t0=0.0, t1=t1, dt=0.01, save=plan, solver="tsit5", rtol=1e-8, atol=1e-10)
ts = np.asarray(res["y"].times.values)
ys = np.asarray(res["y"].values.data)[:, 0]
analytic = np.exp(-0.05 * ts * ts)
np.testing.assert_allclose(ys, analytic, rtol=1e-5)
print(f"final y={ys[-1]:.6f}, analytic={analytic[-1]:.6f}")


## Seasonal contact, step intervention, sigmoidal ramp

Each expression is asserted at known points, not only plotted.


In [ ]:
from summer4 import EntryFlow

seasonal = linear(Time(), (0.0, 90.0, 180.0, 270.0, 360.0), (0.2, 0.4, 0.2, 0.4, 0.2))
intervention = step(Time(), (100.0,), (1.0, 0.3))
ramp = sigmoidal(Time(), (50.0, 90.0), (0.1, 0.5), sharpness=16.0)


def rate_at(expr, t):
    m = FlowModel(pmap)
    m.add_flow(EntryFlow("in", state["Y"], expr))
    return float(np.asarray(m.compile().vector_field(t, y0, {}).data)[0])


np.testing.assert_allclose(rate_at(seasonal, 0.0), 0.2)
np.testing.assert_allclose(rate_at(seasonal, 90.0), 0.4)
np.testing.assert_allclose(rate_at(seasonal, 45.0), 0.3)
np.testing.assert_allclose(rate_at(intervention, 99.0), 1.0)
np.testing.assert_allclose(rate_at(intervention, 100.0), 0.3)
np.testing.assert_allclose(rate_at(ramp, 50.0), 0.1, atol=1e-5)
np.testing.assert_allclose(rate_at(ramp, 90.0), 0.5, atol=1e-5)
assert rate_at(ramp, 70.0) > 0.1 and rate_at(ramp, 70.0) < 0.5
print("seasonal / step / sigmoidal checks ok")


## Dated series via `Data`

Load calendar observations through an `Epoch`, then drive a flow rate with
`data.interp()`. Outside the observed window the interpolator clamps.


In [ ]:
from datetime import date

import pandas as pd

from summer4 import Epoch
from summer4.data import Data

epoch = Epoch(date(2020, 1, 1))
dates = pd.to_datetime(["2020-01-01", "2020-01-31", "2020-03-01", "2020-04-01"])
series = pd.Series([0.05, 0.2, 0.15, 0.05], index=dates)
data = Data.from_series(series, epoch)
rate = data.interp(kind="linear")

dm = FlowModel(pmap)
dm.add_flow(EntryFlow("seed", state["Y"], rate))
dcm = dm.compile()
dplan = SavePlan(
    requests={"y": SaveRequest(Compartments())},
    ts=np.linspace(float(data.times[0]), float(data.times[-1]), 61),
)
dres = dcm.run({}, PropertyData.wrap(pmap, np.array([0.0])), t0=float(data.times[0]), t1=float(data.times[-1]), dt=1.0, save=dplan, solver="tsit5", epoch=epoch)
# Round-trip: interpolant matches observations on their dates.
for t, v in zip(data.times, data.values, strict=True):
    np.testing.assert_allclose(rate_at(rate, float(t)), float(v))
# Clamp past the last observation.
np.testing.assert_allclose(rate_at(rate, float(data.times[-1]) + 30.0), float(data.values[-1]))
cal = pd.DatetimeIndex(epoch.from_model(np.asarray(dres["y"].times.values)))
print(f"ran {cal[0].date()} → {cal[-1].date()}, final y={float(np.asarray(dres['y'].values.data)[-1, 0]):.3f}")


In [ ]:
sir = Property("state", ("S", "I", "R"))
spmap = PropertyMap.from_property(sir)
contact = linear(Time(), (0.0, 180.0), (0.3, 0.15))
sm = FlowModel(spmap)
sm.add_flow(TransitionFlow("infection", sir["S"], sir["I"], contact))
sm.add_flow(TransitionFlow("recovery", sir["I"], sir["R"], 0.1))
scm = sm.compile()
sy0 = PropertyData.wrap(spmap, np.array([999.0, 1.0, 0.0]))
splan = SavePlan(requests={"I": SaveRequest(Compartments(where=sir["I"]))})
sres = scm.run({}, sy0, t0=0.0, t1=180.0, dt=1.0, save=splan, solver="tsit5")
i_peak = float(np.max(np.asarray(sres["I"].values.data)))
assert i_peak > 1.0
print(f"peak I under seasonal contact: {i_peak:.1f}")


## Parametric breakpoints

Knot *times* calibrate the same way knot heights do. A step with a
`FieldRef` start time moves when you change the param. Under `jax.jit`,
`value_and_grad` through a *linear* knot time and height matches finite
differences (step is discontinuous, so grads use linear).


In [ ]:
from typing import Any, NamedTuple

import jax
import jax.numpy as jnp

from summer4 import derived_refs


class Start(NamedTuple):
    t_start: float


class Both(NamedTuple):
    t_mid: float
    height: float


def rate_at_params(expr, t, params):
    m = FlowModel(pmap)
    m.add_flow(EntryFlow("in", state["Y"], expr))
    return float(np.asarray(m.compile().vector_field(t, y0, params).data)[0])


srefs = derived_refs(Start)
# Fixed y, parametric x: step down at a calibratable time.
intervention_x = step(Time(), (srefs.t_start,), (1.0, 0.3))
np.testing.assert_allclose(rate_at_params(intervention_x, 99.0, Start(t_start=100.0)), 1.0)
np.testing.assert_allclose(rate_at_params(intervention_x, 100.0, Start(t_start=100.0)), 0.3)
np.testing.assert_allclose(rate_at_params(intervention_x, 100.0, Start(t_start=120.0)), 1.0)

brefs = derived_refs(Both)
both = linear(Time(), (0.0, brefs.t_mid, 10.0), (0.0, brefs.height, 0.0))
np.testing.assert_allclose(
    rate_at_params(both, 5.0, Both(t_mid=5.0, height=2.2)), 2.2, atol=1e-5
)
# jitted vector field matches eager for parametric x+y
jm = FlowModel(pmap)
jm.add_flow(EntryFlow("in", state["Y"], both))
jcm = jm.compile()
eager = np.asarray(jcm.vector_field(5.0, y0, Both(t_mid=5.0, height=2.2)).data)
jitted = np.asarray(jax.jit(jcm.vector_field)(5.0, y0, Both(t_mid=5.0, height=2.2)).data)
np.testing.assert_allclose(jitted, eager, rtol=1e-6)

# Grad through linear knot time + height under jax.jit(value_and_grad).
peak = linear(Time(), (0.0, brefs.t_mid, 20.0), (0.0, brefs.height, 0.0))
im = FlowModel(pmap)
im.add_flow(EntryFlow("in", state["Y"], peak))
icm = im.compile()
iy0 = PropertyData.wrap(pmap, np.array([0.0]))
iplan = SavePlan(requests={"y": SaveRequest(Compartments())}, ts=np.array([10.0]))


def loss(params: Both) -> Any:
    res = icm.run(
        params,
        iy0,
        t0=0.0,
        t1=10.0,
        dt=0.5,
        save=iplan,
        solver="euler",
    )
    return jnp.sum(jnp.asarray(res["y"].values.data))


value_and_grad = jax.jit(jax.value_and_grad(loss))
base = Both(t_mid=10.0, height=1.0)
val, grads = value_and_grad(base)
eps = 1e-3
fd_t = (
    float(loss(Both(t_mid=10.0 + eps, height=1.0))) - float(loss(Both(t_mid=10.0 - eps, height=1.0)))
) / (2 * eps)
fd_h = (
    float(loss(Both(t_mid=10.0, height=1.0 + eps))) - float(loss(Both(t_mid=10.0, height=1.0 - eps)))
) / (2 * eps)
assert np.isfinite(float(val))
np.testing.assert_allclose(float(grads.t_mid), fd_t, rtol=5e-2, atol=1e-2)
np.testing.assert_allclose(float(grads.height), fd_h, rtol=5e-2, atol=1e-2)
print(
    f"jit value_and_grad: t_mid={float(grads.t_mid):.4f} (fd={fd_t:.4f}), "
    f"height={float(grads.height):.4f} (fd={fd_h:.4f})"
)
